# SimSat Gemma-4 v11 — T4 Eval on Reviewed Pool

Run a full v11 fine-tune evaluation against the freshly-reviewed SimSat trace pool, on a Colab T4 (16GB VRAM, much faster than RTX 2080 4-bit).

**Inputs you provide:**
1. `simsat_eval_pool.jsonl` — generated locally via `python scripts/export_eval_data.py --output simsat_eval_pool.jsonl`. Upload via the file widget in cell 3.
2. (optional) `simsat_eval_pool.tar.gz` — same JSONL bundled with thumbnails. Use if you want image-conditioned eval.

**Output:** a results table + per-scenario breakdown + downloadable CSV.

Runtime: T4 (`Runtime > Change runtime type > T4 GPU`). Total wall clock for ~150 cases: 15-25 min.

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.46.3 peft==0.15.2 accelerate==1.1.1 bitsandbytes==0.45.0 huggingface_hub==0.27.0 pillow

## 2. Set HuggingFace token
Either set as a Colab Secret named `HF_TOKEN` (Colab `🔑 icon` in left sidebar), or paste below.

In [ ]:
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    HF_TOKEN = input('Paste HF token (or set Colab Secret HF_TOKEN): ').strip()
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

## 3. Upload the eval pool JSONL
Either: drag-and-drop `simsat_eval_pool.jsonl` into the file panel on the left, OR run the next cell to use Colab's upload widget.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select simsat_eval_pool.jsonl
EVAL_PATH = list(uploaded.keys())[0]
print(f'Eval pool: {EVAL_PATH}')

## 4. Load v11 adapter on top of Gemma-4-E2B-IT

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = 'google/gemma-4-e2b-it'
ADAPTER = 'HumanAIConvention/simsat-gemma4-v11'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)

print('Loading base model (float16)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='cuda',
    token=HF_TOKEN,
)

print(f'Loading LoRA adapter: {ADAPTER}')
model = PeftModel.from_pretrained(model, ADAPTER, token=HF_TOKEN)
model.eval()
print(f'Model ready on {next(model.parameters()).device} ({next(model.parameters()).dtype}).')

## 5. Run inference over the eval pool

In [ ]:
import json, re, time
from collections import Counter

with open(EVAL_PATH) as f:
    rows = [json.loads(l) for l in f if l.strip()]
print(f'Eval pool: {len(rows)} cases')

def parse_action(text: str):
    """Extract recommended_action + numeric usefulness signal from JSON output."""
    m = re.search(r'\{.*?\}', text, re.DOTALL)
    if not m:
        return None, None, None
    try:
        obj = json.loads(m.group())
    except Exception:
        return None, None, None
    action = obj.get('recommended_action')
    usable = obj.get('usable_observation')
    # composite usefulness from scene_match * salience (avoid divides)
    salience = obj.get('salience_score', 0) or 0
    scene = obj.get('scene_match_score', 0) or 0
    score = max(0.0, min(1.0, 0.5 * (salience + scene)))
    return action, bool(usable) if usable is not None else None, score

results = []
t0 = time.time()
for i, r in enumerate(rows, start=1):
    msgs = [
        {'role': 'system', 'content': r['system_prompt']},
        {'role': 'user',   'content': r['user_prompt']},
    ]
    inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    pred_action, pred_useful, pred_score = parse_action(text)
    results.append({
        'trace_id': r['trace_id'],
        'scenario': r['scenario_pack'],
        'target': r['target_label'],
        'cloud': r['metadata'].get('cloud_cover_pct'),
        'pred_action': pred_action,
        'pred_useful': pred_useful,
        'pred_score': pred_score,
        'op_action': r['operator_action'],
        'op_useful': r['operator_useful'],
        'op_score': r['operator_usefulness'],
        'parse_ok': pred_action is not None,
        'raw': text[:300],
    })
    if i % 10 == 0:
        elapsed = time.time() - t0
        print(f'  [{i}/{len(rows)}] elapsed={elapsed:.0f}s · {elapsed/i:.1f}s/case')

print(f'\nDone in {time.time()-t0:.0f}s · {len(results)} cases')

## 6. Summary, per-pack breakdown, and baselines

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

# Filter to parsable rows
df_ok = df[df.parse_ok].copy()
n_ok, n_total = len(df_ok), len(df)
print(f'Parse rate: {n_ok}/{n_total} = {n_ok/n_total:.2%}')

# Headline
exact = (df_ok.pred_action == df_ok.op_action).mean()
useful = (df_ok.pred_useful == df_ok.op_useful).mean()
mae = (df_ok.pred_score.fillna(0) - df_ok.op_score.fillna(0)).abs().mean()
print(f'\nHeadline (over {n_ok} parsed):')
print(f'  Exact action agreement: {exact:.3f}')
print(f'  Useful agreement:       {useful:.3f}')
print(f'  Score MAE:              {mae:.3f}')

# Per-pack
print('\nPer-pack:')
for pack, sub in df_ok.groupby('scenario'):
    e = (sub.pred_action == sub.op_action).mean()
    print(f'  {pack:30s} N={len(sub):3d}  exact={e:.3f}')

# Baselines
from collections import Counter
op_dist = Counter(df_ok.op_action)
majority_class, majority_count = op_dist.most_common(1)[0]
print(f'\nOperator action distribution: {dict(op_dist)}')
print(f'Always-majority ({majority_class}) baseline: {majority_count/n_ok:.3f}')
print(f'Random uniform (4-class) baseline: 0.250')

df_ok.to_csv('v11_eval_results.csv', index=False)
print('\nSaved v11_eval_results.csv')

## 7. Confusion matrix + download

In [ ]:
import pandas as pd
actions = ['accept', 'refine', 'defer', 'skip']
cm = pd.crosstab(df_ok.op_action, df_ok.pred_action).reindex(index=actions, columns=actions, fill_value=0)
print('Confusion matrix (rows = operator, columns = v11 prediction):')
print(cm)

from google.colab import files
files.download('v11_eval_results.csv')